# 05 — GPT-5.5 factorial completion (revision round 3)

Fills the eight prompt conditions missing for GPT-5.5, so that the 5 x 2
factorial is fully crossed across all three model generations.

Already recorded in `frontier_gpt55.csv` (720 rows): `A_clean`, `E_clean`.

Run order is deliberate — **Stage 1 first, then read the result before
spending on Stage 2**:

| Stage | Conditions | Calls | Answers |
|---|---|---|---|
| 1 | B_clean, C_clean, D_clean | 1080 | Reviewer 3 pt.3 / Reviewer 4 pt.2 — component attribution |
| 2 | A/B/C/D/E hinted | 1800 | Reviewer 3 pt.2 — RQ3 above the capability threshold |

Results append to the existing `frontier_gpt55.csv`; the runner's resume set is
keyed on `(sample_id, condition, run_id)`, so completed conditions are skipped.

In [1]:
# --- environment (identical to notebook 03) ---
!pip -q install openai
from google.colab import drive; drive.mount('/content/drive')

import sys, shutil
from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/LLM_Security_Paper')   # SARD corpus root
WORK = ROOT / 'revision_2026'                              # outputs live here
WORK.mkdir(exist_ok=True)

sys.path.insert(0, str(WORK))          # vulnbench.py lives in WORK
import vulnbench as vb

BENCH = pd.read_csv(WORK / 'benchmark_360_metadata.csv')
print(len(BENCH), 'samples |', BENCH.true_label.value_counts().to_dict())

Mounted at /content/drive
360 samples | {'Safe': 180, 'Vulnerable': 180}


In [2]:
import os, getpass
os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')
from openai import OpenAI
client = OpenAI()

OpenAI API key: ··········


## Preflight — back up, then confirm exactly what is missing

The backup matters: every subsequent cell **appends to the validated
`frontier_gpt55.csv`**. If a run misbehaves, the untouched copy is the way back
to the numbers currently in the manuscript.

In [ ]:
MAIN = WORK / 'frontier_gpt55.csv'
MODEL = 'gpt-5.5-2026-04-23'
ALL_CONDITIONS = [(v, h) for v in 'ABCDE' for h in ('clean', 'hinted')]

# --- back up before any append ---
BACKUP = WORK / 'frontier_gpt55_BACKUP_pre_stage1.csv'
if MAIN.exists() and not BACKUP.exists():
    shutil.copy2(MAIN, BACKUP)
    print('backup written ->', BACKUP.name)
elif BACKUP.exists():
    print('backup already exists ->', BACKUP.name)

# --- what is done, what is missing ---
prev = pd.read_csv(MAIN)
done = sorted(prev.condition.unique())
missing = [f'{v}_{h}' for v, h in ALL_CONDITIONS if f'{v}_{h}' not in done]

print(f'\nrows on file : {len(prev)}')
print(f'complete     : {done}')
print(f'missing      : {missing}  ({len(missing)} conditions x {len(BENCH)} = '
      f'{len(missing) * len(BENCH)} calls)')

# integrity: every recorded condition should have exactly one row per sample
counts = prev.groupby('condition').sample_id.nunique()
assert (counts == len(BENCH)).all(), f'incomplete condition on file: {counts.to_dict()}'
assert prev.prediction.isna().sum() == 0, 'unparsed predictions already on file'
print('\nintegrity OK — every recorded condition covers all 360 samples')

backup written -> frontier_gpt55_BACKUP_pre_stage1.csv

rows on file : 720
complete     : ['A_clean', 'E_clean']
missing      : ['A_hinted', 'B_clean', 'B_hinted', 'C_clean', 'C_hinted', 'D_clean', 'D_hinted', 'E_hinted']  (8 conditions x 360 = 2880 calls)

integrity OK — every recorded condition covers all 360 samples


## Cost probe — measure before committing

18 calls against a separate file, so the probe never enters the analysis set.
The runner prints real token counts; the extrapolation below turns those into a
budget figure for each stage.

In [ ]:
probe = BENCH.groupby('true_label', group_keys=False).sample(3, random_state=11)

rp = vb.run_experiment(
    bench        = probe,
    dataset_root = ROOT,
    conditions   = [('B', 'clean'), ('C', 'clean'), ('D', 'clean')],
    model        = MODEL,
    temperature  = None,                      # GPT-5.5 rejects a custom value
    out_csv      = WORK / 'probe_gpt55_stage1.csv',
    client       = client,
)

# the runner prints 'this session cost ~ $X' — read cost per call from it
n_probe = len(rp)
print(f'\nprobe calls: {n_probe}')
print('Take the printed session cost, divide by', n_probe, '= cost per call.')
print('Then:  Stage 1 = per_call x 1080   |   Stage 2 = per_call x 1800')
print('\nprobe verdicts:')
print(pd.crosstab(rp.true_label, rp.prediction))

18 calls to make with gpt-5.5-2026-04-23
  18/18  |  tokens in/out 5220/8643  |  running cost $0.285

done — 18 rows in /content/drive/MyDrive/LLM_Security_Paper/revision_2026/probe_gpt55_stage1.csv
this session cost ≈ $0.285

probe calls: 18
Take the printed session cost, divide by 18 = cost per call.
Then:  Stage 1 = per_call x 1080   |   Stage 2 = per_call x 1800

probe verdicts:
prediction  Safe  Vulnerable
true_label                  
Safe           1           8
Vulnerable     3           6


## Stage 1 — B_clean, C_clean, D_clean  (1080 calls)

These three isolate persona, taint-analysis instruction and step-by-step
reasoning as single factors. Together with the existing `A_clean` and
`E_clean`, they make the conservative shift attributable to a component rather
than to the bundle.

Safe to interrupt: partial progress is on disk and this cell resumes from it.

In [ ]:
res_s1 = vb.run_experiment(
    bench        = BENCH,
    dataset_root = ROOT,
    conditions   = [('B', 'clean'), ('C', 'clean'), ('D', 'clean')],
    model        = MODEL,
    temperature  = None,
    out_csv      = MAIN,
    client       = client,
)
print('\nrows now on file:', len(res_s1))
print(res_s1.condition.value_counts().sort_index())

resuming — 720 calls already recorded
1080 calls to make with gpt-5.5-2026-04-23
  50/1080  |  tokens in/out 9941/19034  |  running cost $0.621
  100/1080  |  tokens in/out 20368/59349  |  running cost $1.882
  150/1080  |  tokens in/out 34279/82816  |  running cost $2.656
  200/1080  |  tokens in/out 49510/108652  |  running cost $3.507
  250/1080  |  tokens in/out 64195/132626  |  running cost $4.300
  300/1080  |  tokens in/out 73999/158065  |  running cost $5.112
  350/1080  |  tokens in/out 84822/187243  |  running cost $6.041
  400/1080  |  tokens in/out 96543/204019  |  running cost $6.603
  450/1080  |  tokens in/out 108836/223305  |  running cost $7.243
  500/1080  |  tokens in/out 123632/241607  |  running cost $7.866
  550/1080  |  tokens in/out 141281/256437  |  running cost $8.400
  600/1080  |  tokens in/out 159092/274298  |  running cost $9.024
  650/1080  |  tokens in/out 170948/293637  |  running cost $9.664
  700/1080  |  tokens in/out 183191/311487  |  running cost $

## Stage 1 readout — does the mechanism survive?

Two questions decide whether Stage 2 is worth funding:

1. **Component attribution.** Does one single factor reproduce the shift seen in
   `E_clean` (recall 0.800, MCC 0.540 against `A_clean` at 0.900 / 0.612)?
2. **Blind spot.** Does the escaping-vs-coercion gap hold across the new
   conditions, or was 94.3% vs 21.2% specific to the bundle?

In [3]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/LLM_Security_Paper')
WORK = ROOT / 'revision_2026'
sys.path.insert(0, str(WORK))

MAIN = WORK / 'frontier_gpt55.csv'
MODEL = 'gpt-5.5-2026-04-23'
ALL_CONDITIONS = [(v, h) for v in 'ABCDE' for h in ('clean', 'hinted')]

BENCH = pd.read_csv(WORK / 'benchmark_360_metadata.csv')
print(len(BENCH), 'samples |', BENCH.true_label.value_counts().to_dict())
print('rows on file:', len(pd.read_csv(MAIN)))

360 samples | {'Safe': 180, 'Vulnerable': 180}
rows on file: 1800


In [ ]:
import numpy as np

df = pd.read_csv(MAIN)
df = df[df.run_id == 1]

def metrics(g):
    tp = ((g.true_label=='Vulnerable') & (g.prediction=='Vulnerable')).sum()
    fn = ((g.true_label=='Vulnerable') & (g.prediction=='Safe')).sum()
    tn = ((g.true_label=='Safe')       & (g.prediction=='Safe')).sum()
    fp = ((g.true_label=='Safe')       & (g.prediction=='Vulnerable')).sum()
    den = np.sqrt(float((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))) or np.nan
    return pd.Series({
        'TP': tp, 'FN': fn, 'TN': tn, 'FP': fp,
        'Recall':      round(tp/(tp+fn), 3) if tp+fn else np.nan,
        'Specificity': round(tn/(tn+fp), 3) if tn+fp else np.nan,
        'BalAcc':      round((tp/(tp+fn) + tn/(tn+fp))/2, 3),
        'MCC':         round((tp*tn - fp*fn)/den, 3),
    })

print('=== all recorded GPT-5.5 conditions ===')
print(df.groupby('condition').apply(metrics, include_groups=False).to_string())

=== all recorded GPT-5.5 conditions ===
              TP    FN     TN    FP  Recall  Specificity  BalAcc    MCC
condition                                                              
A_clean    162.0  18.0  126.0  54.0   0.900        0.700   0.800  0.612
B_clean    156.0  24.0  127.0  53.0   0.867        0.706   0.786  0.580
C_clean    147.0  33.0  132.0  48.0   0.817        0.733   0.775  0.552
D_clean    140.0  40.0  132.0  48.0   0.778        0.733   0.756  0.512
E_clean    144.0  36.0  133.0  47.0   0.800        0.739   0.769  0.540


In [ ]:
# --- sanitisation blind spot, per condition ---
# Does the escaping/coercion asymmetry hold outside the A-vs-E pair?
safe = df[df.true_label == 'Safe'].copy()

if 'sanitizer' in safe.columns and safe.sanitizer.notna().any():
    tab = (safe.assign(correct=lambda d: d.prediction == 'Safe')
               .pivot_table(index='sanitizer', columns='condition',
                            values='correct', aggfunc='mean'))
    print('cleared rate on safe samples, by sanitiser mechanism:')
    print((tab * 100).round(1).to_string())
    print('\nsample counts per mechanism:')
    print(safe[safe.condition == safe.condition.iloc[0]].sanitizer.value_counts().to_string())
else:
    print('no sanitizer column populated — join against benchmark_360_metadata.csv')

cleared rate on safe samples, by sanitiser mechanism:
condition                                   A_clean  B_clean  C_clean  D_clean  E_clean
sanitizer                                                                              
CAST-cast_float                               100.0    100.0    100.0    100.0     83.3
CAST-cast_float_sort_of                       100.0    100.0    100.0    100.0    100.0
CAST-cast_int                                 100.0    100.0    100.0    100.0    100.0
CAST-cast_int_sort_of                         100.0    100.0    100.0    100.0    100.0
CAST-cast_int_sort_of2                         85.7     85.7     85.7     85.7     85.7
CAST-func_settype_float                       100.0    100.0    100.0    100.0    100.0
CAST-func_settype_int                          83.3     83.3     83.3     83.3     83.3
func_FILTER-CLEANING-magic_quotes_filter        0.0      0.0     14.3      0.0     14.3
func_FILTER-CLEANING-number_float_filter      100.0    100.0    10

## Stage 2 — the five HINTED conditions  (1800 calls)

**Run only after reading the Stage 1 readout.** These complete the factorial and
give RQ3 observations above the capability threshold, where it currently has
none.

In [4]:
res_s2 = vb.run_experiment(
    bench        = BENCH,
    dataset_root = ROOT,
    conditions   = [(v, 'hinted') for v in 'ABCDE'],
    model        = MODEL,
    temperature  = None,
    out_csv      = MAIN,
    client       = client,
)
print('\nrows now on file:', len(res_s2))

resuming — 1800 calls already recorded
1800 calls to make with gpt-5.5-2026-04-23
  50/1800  |  tokens in/out 9991/13244  |  running cost $0.447
  100/1800  |  tokens in/out 20468/36369  |  running cost $1.193
  150/1800  |  tokens in/out 34399/55879  |  running cost $1.848
  200/1800  |  tokens in/out 49630/82722  |  running cost $2.730
  250/1800  |  tokens in/out 64335/104386  |  running cost $3.453
  300/1800  |  tokens in/out 74239/117422  |  running cost $3.894
  350/1800  |  tokens in/out 85162/146326  |  running cost $4.816
  400/1800  |  tokens in/out 95903/163140  |  running cost $5.374
  450/1800  |  tokens in/out 106946/188800  |  running cost $6.199
  500/1800  |  tokens in/out 120472/214068  |  running cost $7.024
  550/1800  |  tokens in/out 136821/236434  |  running cost $7.777
  600/1800  |  tokens in/out 153332/276764  |  running cost $9.070
  650/1800  |  tokens in/out 163988/291177  |  running cost $9.555
  700/1800  |  tokens in/out 175031/316584  |  running cost $

## Final verification — the factorial is closed

In [5]:
final = pd.read_csv(MAIN)
f1 = final[final.run_id == 1]

expected = {f'{v}_{h}' for v, h in ALL_CONDITIONS}
present  = set(f1.condition.unique())

print('conditions present :', len(present), '/ 10')
print('still missing      :', sorted(expected - present) or 'none')
print('rows               :', len(f1), '(expect 3600)')
print('unparsed responses :', int(f1.prediction.isna().sum()))

counts = f1.groupby('condition').sample_id.nunique()
print('\nper-condition sample coverage:')
print(counts.to_string())
assert (counts == 360).all(), 'a condition does not cover all 360 samples'

print('\n=== full metric set, GPT-5.5, 10 conditions ===')
print(f1.groupby('condition').apply(metrics, include_groups=False).to_string())

conditions present : 10 / 10
still missing      : none
rows               : 3600 (expect 3600)
unparsed responses : 0

per-condition sample coverage:
condition
A_clean     360
A_hinted    360
B_clean     360
B_hinted    360
C_clean     360
C_hinted    360
D_clean     360
D_hinted    360
E_clean     360
E_hinted    360

=== full metric set, GPT-5.5, 10 conditions ===


NameError: name 'metrics' is not defined

In [6]:
import numpy as np

def metrics(g):
    tp = ((g.true_label=='Vulnerable') & (g.prediction=='Vulnerable')).sum()
    fn = ((g.true_label=='Vulnerable') & (g.prediction=='Safe')).sum()
    tn = ((g.true_label=='Safe')       & (g.prediction=='Safe')).sum()
    fp = ((g.true_label=='Safe')       & (g.prediction=='Vulnerable')).sum()
    den = np.sqrt(float((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))) or np.nan
    return pd.Series({
        'TP': tp, 'FN': fn, 'TN': tn, 'FP': fp,
        'Recall':      round(tp/(tp+fn), 3) if tp+fn else np.nan,
        'Specificity': round(tn/(tn+fp), 3) if tn+fp else np.nan,
        'BalAcc':      round((tp/(tp+fn) + tn/(tn+fp))/2, 3),
        'MCC':         round((tp*tn - fp*fn)/den, 3),
    })

f1 = pd.read_csv(MAIN)
f1 = f1[f1.run_id == 1]
print(f1.groupby('condition').apply(metrics, include_groups=False).to_string())

              TP     FN     TN    FP  Recall  Specificity  BalAcc    MCC
condition                                                               
A_clean    162.0   18.0  126.0  54.0   0.900        0.700   0.800  0.612
A_hinted   104.0   76.0  163.0  17.0   0.578        0.906   0.742  0.512
B_clean    156.0   24.0  127.0  53.0   0.867        0.706   0.786  0.580
B_hinted    98.0   82.0  163.0  17.0   0.544        0.906   0.725  0.483
C_clean    147.0   33.0  132.0  48.0   0.817        0.733   0.775  0.552
C_hinted   130.0   50.0  155.0  25.0   0.722        0.861   0.792  0.589
D_clean    140.0   40.0  132.0  48.0   0.778        0.733   0.756  0.512
D_hinted    79.0  101.0  166.0  14.0   0.439        0.922   0.681  0.412
E_clean    144.0   36.0  133.0  47.0   0.800        0.739   0.769  0.540
E_hinted   113.0   67.0  161.0  19.0   0.628        0.894   0.761  0.542


In [7]:
# --- optional drift check: replicate A_clean as run_id 2 on a 48-sample subset ---
# The new conditions were executed in a different session from A_clean/E_clean.
# This confirms the snapshot behaves consistently before old and new conditions
# are compared in the same table. ~48 calls.
sub = BENCH.groupby(['cwe', 'true_label'], group_keys=False).sample(8, random_state=42)

drift = vb.run_experiment(
    bench        = sub,
    dataset_root = ROOT,
    conditions   = [('A', 'clean')],
    model        = MODEL,
    temperature  = None,
    run_ids      = (2,),                      # run_id 2 -> does not collide
    out_csv      = WORK / 'drift_check_A_clean.csv',
    client       = client,
)

orig = pd.read_csv(MAIN)
orig = orig[(orig.condition == 'A_clean') & (orig.run_id == 1) &
            (orig.sample_id.isin(sub.sample_id))]
merged = drift.merge(orig[['sample_id', 'prediction']], on='sample_id',
                     suffixes=('_new', '_orig'))
agree = (merged.prediction_new == merged.prediction_orig).sum()
print(f'\nverdict agreement with the original run: {agree}/{len(merged)} '
      f'({agree/len(merged):.1%})')

48 calls to make with gpt-5.5-2026-04-23
  48/48  |  tokens in/out 10703/20787  |  running cost $0.677

done — 48 rows in /content/drive/MyDrive/LLM_Security_Paper/revision_2026/drift_check_A_clean.csv
this session cost ≈ $0.677

verdict agreement with the original run: 44/48 (91.7%)
